In [2]:
import pandas as pd 

In [3]:
# read the big file in chunks and kepp only 2017/2018 rows
chunks=[]
for chunk in pd.read_csv("charts.csv",chunksize=500000):
    chunk["date"] = pd.to_datetime(chunk["date"])
    chunk = chunk[chunk["date"].dt.year.isin([2017,2018])]
    chunks.append(chunk)

df = pd.concat(chunks)

In [4]:
#pull the spotify track id out of the url 

df["track_id"] = df["url"].str.extract(r"/track/(\w+)")
df = df.dropna(subset=["track_id"])


In [5]:
#basic clean up

df["title"] = df["title"].str.strip()
df["artist"] = df["artist"].str.strip()

In [6]:
#rename columns to match it in SQL

df = df.rename(columns={
    "title":"track_title",
    "date":"chart_date",
    "artist":"artist_name",
    "chart":"chart_type",
    "url":"source_url",
})

In [7]:
# renaming the cleaned file

df.to_csv("charts_2017_18_cleaned.csv",index=False)

In [8]:
df = pd.read_csv("charts_2017_18_cleaned.csv")

In [9]:
# understand the data

In [10]:
df.head()

,track_title,rank,chart_date,artist_name,source_url,region,chart_type,trend,streams,track_id
0,Chantaje (feat. Maluma),1,2017-01-01,Shakira,https://open.spotify.com/track/6mICuAdrwEjh6Y6...,Argentina,top200,SAME_POSITION,253019.0,6mICuAdrwEjh6Y6lroV2Kg
1,Vente Pa' Ca (feat. Maluma),2,2017-01-01,Ricky Martin,https://open.spotify.com/track/7DM4BPaS7uofFul...,Argentina,top200,MOVE_UP,223988.0,7DM4BPaS7uofFul3ywMe46
2,Reggaetón Lento (Bailemos),3,2017-01-01,CNCO,https://open.spotify.com/track/3AEZUABDXNtecAO...,Argentina,top200,MOVE_DOWN,210943.0,3AEZUABDXNtecAOSC1qTfo
3,Safari,4,2017-01-01,"J Balvin, Pharrell Williams, BIA, Sky",https://open.spotify.com/track/6rQSrBHf7HlZjtc...,Argentina,top200,SAME_POSITION,173865.0,6rQSrBHf7HlZjtcMZ4S4bO
4,Shaky Shaky,5,2017-01-01,Daddy Yankee,https://open.spotify.com/track/58IL315gMSTD37D...,Argentina,top200,MOVE_UP,153956.0,58IL315gMSTD37DOZPJ2hf


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9264161 entries, 0 to 9264160
Data columns (total 10 columns):
 #   Column       Dtype  
---  ------       -----  
 0   track_title  str    
 1   rank         int64  
 2   chart_date   str    
 3   artist_name  str    
 4   source_url   str    
 5   region       str    
 6   chart_type   str    
 7   trend        str    
 8   streams      float64
 9   track_id     str    
dtypes: float64(1), int64(1), str(8)
memory usage: 706.8 MB


In [12]:
df.shape

(9264161, 10)

In [13]:
df.dtypes

track_title        str
rank             int64
chart_date         str
artist_name        str
source_url         str
region             str
chart_type         str
trend              str
streams        float64
track_id           str
dtype: object

In [14]:
df.describe()

,rank,streams
count,9.264161e+06,7.125835e+06
mean,7.896825e+01,5.328280e+04
std,5.865956e+01,2.048518e+05
min,1.000000e+00,1.001000e+03
25%,2.800000e+01,3.211000e+03
50%,6.300000e+01,9.151000e+03
75%,1.280000e+02,3.113300e+04
max,2.000000e+02,1.138152e+07


In [15]:
# finding duplicates and nulls

In [16]:
df.isnull().sum().sort_values(ascending=False)

streams        2138326
track_title          0
rank                 0
chart_date           0
source_url           0
artist_name          0
region               0
chart_type           0
trend                0
track_id             0
dtype: int64

In [17]:
df.duplicated().sum()

np.int64(0)

In [18]:
# finding nulls based on charts 

In [19]:
df[df["streams"].isnull()]["chart_type"].value_counts()

chart_type
viral50    2138326
Name: count, dtype: int64

In [20]:
# understanding each unique data points in the data set 

In [21]:
df["region"].unique()

<StringArray>
[           'Argentina',            'Australia',               'Brazil',
              'Austria',              'Belgium',             'Colombia',
              'Bolivia',              'Denmark',             'Bulgaria',
               'Canada',                'Chile',           'Costa Rica',
       'Czech Republic',              'Finland',   'Dominican Republic',
              'Ecuador',          'El Salvador',              'Estonia',
               'France',              'Germany',               'Global',
               'Greece',            'Guatemala',             'Honduras',
            'Hong Kong',              'Hungary',              'Iceland',
            'Indonesia',              'Ireland',                'Italy',
                'Japan',               'Latvia',            'Lithuania',
             'Malaysia',           'Luxembourg',               'Mexico',
          'Netherlands',          'New Zealand',            'Nicaragua',
               'Norway',             

In [22]:
df["chart_type"].unique()

<StringArray>
['top200', 'viral50']
Length: 2, dtype: str

In [23]:
df["trend"].value_counts()

trend
MOVE_DOWN        3918639
MOVE_UP          3485739
SAME_POSITION    1088774
NEW_ENTRY         771009
Name: count, dtype: int64

In [24]:
# the date was stored as a string so had to convert that

df = pd.read_csv("charts_2017_18_cleaned.csv" , parse_dates=["chart_date"])

In [25]:
df.dtypes

track_title               str
rank                    int64
chart_date     datetime64[us]
artist_name               str
source_url                str
region                    str
chart_type                str
trend                     str
streams               float64
track_id                  str
dtype: object

In [26]:
df["chart_date"].dt.year.unique()

array([2017, 2018], dtype=int32)

In [27]:
# just wanted to confirm that there are 730 days from 2017 and 2018
df["chart_date"].unique()

<DatetimeArray>
['2017-01-01 00:00:00', '2017-01-02 00:00:00', '2018-03-01 00:00:00',
 '2018-03-02 00:00:00', '2017-01-03 00:00:00', '2017-02-01 00:00:00',
 '2017-08-01 00:00:00', '2017-08-02 00:00:00', '2018-03-03 00:00:00',
 '2017-02-03 00:00:00',
 ...
 '2017-07-24 00:00:00', '2017-07-27 00:00:00', '2017-07-26 00:00:00',
 '2017-07-28 00:00:00', '2017-07-29 00:00:00', '2017-07-30 00:00:00',
 '2017-07-31 00:00:00', '2017-06-02 00:00:00', '2017-05-30 00:00:00',
 '2017-05-31 00:00:00']
Length: 730, dtype: datetime64[us]

In [28]:
# confirming rank values (1 -200)
df["rank"].min()

np.int64(1)

In [29]:
df["rank"].max()

np.int64(200)

In [30]:
# To confirm that one track holds only one rank per day/region/chart_type
df.duplicated(subset=['track_id','region','chart_date','chart_type']).sum()

np.int64(0)

In [31]:
# making sure that track id only consists of 22 charachters , anything shorter is a regex grabbed from a malformed url
df["track_id"].str.len().value_counts()

track_id
22    9264161
Name: count, dtype: int64

In [32]:
df["artist_name"].unique()

<StringArray>
[                                        'Shakira',
                                    'Ricky Martin',
                                            'CNCO',
           'J Balvin, Pharrell Williams, BIA, Sky',
                                    'Daddy Yankee',
                                 'Sebastian Yatra',
                                          'Rombai',
                                   'Zion & Lennox',
                           'Carlos Vives, Shakira',
                                           'Ozuna',
 ...
                              'The Mountain Goats',
                                    'Hall & Bates',
                               'Lafa Taylor, Aabo',
               'Noisecontrollers, Bass Modulators',
                                   'Kiah Victoria',
 'Pink Guy, Getter, Borgore, Axel Boy, TastyTreat',
                                     'The Butlers',
                                           'Beak>',
                              'Strangely Arou

In [33]:
# rough "base title" - just chop off anything after ( or -
df["title_base"] = df["track_title"].str.split(r"[\(\-]").str[0].str.strip()


In [34]:
# for each base title how many diff track ids are associated to it
dupes = df.groupby("title_base")["track_id"].nunique()
dupes = dupes[dupes>1].sort_values(ascending = False)



In [35]:
type(df["title_base"].iloc[0])

str

In [36]:
dupes.head(30)

title_base
Intro           80
                56
Home            45
Breathe         44
Despacito       42
Alone           36
Closer          35
High            35
Stay            34
Crazy           34
Sorry           32
You             32
Paradise        32
Love            31
Alive           30
Beautiful       29
Bella Ciao      29
Mama            28
Fuego           28
All Night       28
Silent Night    27
Time            27
Gold            27
Run             27
Trouble         26
Baby            26
Runaway         26
Higher          26
Fire            25
Solo            25
Name: track_id, dtype: int64

In [37]:
# validate same title multiple tracks/songs scope :
dupes = df.groupby(["title_base", "artist_name"])["track_id"].nunique()
dupes = dupes[dupes > 1].sort_values(ascending=False)
dupes.head(30)

title_base                       artist_name                                     
Sick Boy                         The Chainsmokers                                    10
Where's My Love                  SYML                                                 9
There's Nothing Holdin' Me Back  Shawn Mendes                                         9
Mercy                            Shawn Mendes                                         9
Mama                             Jonas Blue, William Singe                            9
September Song                   JP Cooper                                            8
Let Me Love You                  DJ Snake, Justin Bieber                              8
Thunder                          Imagine Dragons                                      8
Issues                           Julia Michaels                                       8
Pot                              Thiaguinho                                           8
Don't Kill My Vibe               Sigri

In [38]:
df["streams"] = df["streams"].astype("Int64")

In [39]:
df["streams"].dtype

Int64Dtype()

In [40]:
df.head(10)

,track_title,rank,chart_date,artist_name,source_url,region,chart_type,trend,streams,track_id,title_base
0,Chantaje (feat. Maluma),1,2017-01-01,Shakira,https://open.spotify.com/track/6mICuAdrwEjh6Y6...,Argentina,top200,SAME_POSITION,253019,6mICuAdrwEjh6Y6lroV2Kg,Chantaje
1,Vente Pa' Ca (feat. Maluma),2,2017-01-01,Ricky Martin,https://open.spotify.com/track/7DM4BPaS7uofFul...,Argentina,top200,MOVE_UP,223988,7DM4BPaS7uofFul3ywMe46,Vente Pa' Ca
2,Reggaetón Lento (Bailemos),3,2017-01-01,CNCO,https://open.spotify.com/track/3AEZUABDXNtecAO...,Argentina,top200,MOVE_DOWN,210943,3AEZUABDXNtecAOSC1qTfo,Reggaetón Lento
3,Safari,4,2017-01-01,"J Balvin, Pharrell Williams, BIA, Sky",https://open.spotify.com/track/6rQSrBHf7HlZjtc...,Argentina,top200,SAME_POSITION,173865,6rQSrBHf7HlZjtcMZ4S4bO,Safari
4,Shaky Shaky,5,2017-01-01,Daddy Yankee,https://open.spotify.com/track/58IL315gMSTD37D...,Argentina,top200,MOVE_UP,153956,58IL315gMSTD37DOZPJ2hf,Shaky Shaky
5,Traicionera,6,2017-01-01,Sebastian Yatra,https://open.spotify.com/track/5J1c3M4EldCfNxX...,Argentina,top200,MOVE_DOWN,151140,5J1c3M4EldCfNxXwrwt8mT,Traicionera
6,Cuando Se Pone a Bailar,7,2017-01-01,Rombai,https://open.spotify.com/track/1MpKZi1zTXpERKw...,Argentina,top200,MOVE_DOWN,148369,1MpKZi1zTXpERKwxmOu1PH,Cuando Se Pone a Bailar
7,Otra vez (feat. J Balvin),8,2017-01-01,Zion & Lennox,https://open.spotify.com/track/3QwBODjSEzelZyV...,Argentina,top200,MOVE_DOWN,143004,3QwBODjSEzelZyVjxPOHdq,Otra vez
8,La Bicicleta,9,2017-01-01,"Carlos Vives, Shakira",https://open.spotify.com/track/0sXvAOmXgjR2QUq...,Argentina,top200,MOVE_UP,126389,0sXvAOmXgjR2QUqLK1MltU,La Bicicleta
9,Dile Que Tu Me Quieres,10,2017-01-01,Ozuna,https://open.spotify.com/track/20ZAJdsKB5IGbGj...,Argentina,top200,MOVE_DOWN,112012,20ZAJdsKB5IGbGj4ilRt2o,Dile Que Tu Me Quieres


In [41]:
# drop the column
df = df.drop(columns=["title_base"])

In [42]:
df.head(10)

,track_title,rank,chart_date,artist_name,source_url,region,chart_type,trend,streams,track_id
0,Chantaje (feat. Maluma),1,2017-01-01,Shakira,https://open.spotify.com/track/6mICuAdrwEjh6Y6...,Argentina,top200,SAME_POSITION,253019,6mICuAdrwEjh6Y6lroV2Kg
1,Vente Pa' Ca (feat. Maluma),2,2017-01-01,Ricky Martin,https://open.spotify.com/track/7DM4BPaS7uofFul...,Argentina,top200,MOVE_UP,223988,7DM4BPaS7uofFul3ywMe46
2,Reggaetón Lento (Bailemos),3,2017-01-01,CNCO,https://open.spotify.com/track/3AEZUABDXNtecAO...,Argentina,top200,MOVE_DOWN,210943,3AEZUABDXNtecAOSC1qTfo
3,Safari,4,2017-01-01,"J Balvin, Pharrell Williams, BIA, Sky",https://open.spotify.com/track/6rQSrBHf7HlZjtc...,Argentina,top200,SAME_POSITION,173865,6rQSrBHf7HlZjtcMZ4S4bO
4,Shaky Shaky,5,2017-01-01,Daddy Yankee,https://open.spotify.com/track/58IL315gMSTD37D...,Argentina,top200,MOVE_UP,153956,58IL315gMSTD37DOZPJ2hf
5,Traicionera,6,2017-01-01,Sebastian Yatra,https://open.spotify.com/track/5J1c3M4EldCfNxX...,Argentina,top200,MOVE_DOWN,151140,5J1c3M4EldCfNxXwrwt8mT
6,Cuando Se Pone a Bailar,7,2017-01-01,Rombai,https://open.spotify.com/track/1MpKZi1zTXpERKw...,Argentina,top200,MOVE_DOWN,148369,1MpKZi1zTXpERKwxmOu1PH
7,Otra vez (feat. J Balvin),8,2017-01-01,Zion & Lennox,https://open.spotify.com/track/3QwBODjSEzelZyV...,Argentina,top200,MOVE_DOWN,143004,3QwBODjSEzelZyVjxPOHdq
8,La Bicicleta,9,2017-01-01,"Carlos Vives, Shakira",https://open.spotify.com/track/0sXvAOmXgjR2QUq...,Argentina,top200,MOVE_UP,126389,0sXvAOmXgjR2QUqLK1MltU
9,Dile Que Tu Me Quieres,10,2017-01-01,Ozuna,https://open.spotify.com/track/20ZAJdsKB5IGbGj...,Argentina,top200,MOVE_DOWN,112012,20ZAJdsKB5IGbGj4ilRt2o


In [43]:
# renaming final file 
df.to_csv("charts_2017_2018_cleaned" , index=False)

In [44]:
import os
print(os.path.abspath("charts_2017_2018_final.csv"))

C:\Users\2025\charts_2017_2018_final.csv


In [45]:
df["streams"].dtype

Int64Dtype()

In [46]:
df.head()

,track_title,rank,chart_date,artist_name,source_url,region,chart_type,trend,streams,track_id
0,Chantaje (feat. Maluma),1,2017-01-01,Shakira,https://open.spotify.com/track/6mICuAdrwEjh6Y6...,Argentina,top200,SAME_POSITION,253019,6mICuAdrwEjh6Y6lroV2Kg
1,Vente Pa' Ca (feat. Maluma),2,2017-01-01,Ricky Martin,https://open.spotify.com/track/7DM4BPaS7uofFul...,Argentina,top200,MOVE_UP,223988,7DM4BPaS7uofFul3ywMe46
2,Reggaetón Lento (Bailemos),3,2017-01-01,CNCO,https://open.spotify.com/track/3AEZUABDXNtecAO...,Argentina,top200,MOVE_DOWN,210943,3AEZUABDXNtecAOSC1qTfo
3,Safari,4,2017-01-01,"J Balvin, Pharrell Williams, BIA, Sky",https://open.spotify.com/track/6rQSrBHf7HlZjtc...,Argentina,top200,SAME_POSITION,173865,6rQSrBHf7HlZjtcMZ4S4bO
4,Shaky Shaky,5,2017-01-01,Daddy Yankee,https://open.spotify.com/track/58IL315gMSTD37D...,Argentina,top200,MOVE_UP,153956,58IL315gMSTD37DOZPJ2hf


In [47]:
df.to_pickle("df_checkpoint.pkl")     # save, once, after cleaning is done

In [48]:
df = pd.read_pickle("df_checkpoint.pkl")   # loads instantly, dtypes intact, no re-parsing

In [49]:
df.to_csv("charts_2017_18_final_v2.csv", index=False)